# LendingClub Charge-Off Risk Modeling

This notebook builds an interpretable model that ranks consumer loans by observed charge-off risk. It uses mature 2011–2013 originations so nearly every loan has a known outcome.

In [11]:
from pathlib import Path
import duckdb
import pandas as pd

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

db_path = project_dir / "lendingclub.duckdb"
con = duckdb.connect(str(db_path))

print(f"Database: {db_path}")

Database: /Users/meetshah/Documents/Codex/lendingclub-risk-analysis/lendingclub.duckdb


## 1. Modeling population

The model uses information available at loan origination. LendingClub's grade, subgrade, and interest rate are excluded so the model learns from underlying borrower and loan characteristics rather than reproducing LendingClub's existing risk score. Post-origination payment and recovery fields are also excluded to prevent leakage.

In [12]:
model_df = con.execute("""
    SELECT
        id,
        YEAR(STRPTIME(issue_d, '%b-%Y')) AS issue_year,
        loan_amnt,
        TRIM(term) AS term,
        COALESCE(emp_length, 'Unknown') AS emp_length,
        home_ownership,
        annual_inc,
        verification_status,
        purpose,
        dti,
        (fico_range_low + fico_range_high) / 2.0 AS fico_score,
        delinq_2yrs,
        inq_last_6mths,
        open_acc,
        pub_rec,
        revol_bal,
        revol_util,
        total_acc,
        mort_acc,
        pub_rec_bankruptcies,
        is_default
    FROM loans_mature
""").df()

train_df = model_df[model_df["issue_year"] <= 2012].copy()
test_df = model_df[model_df["issue_year"] == 2013].copy()

split_summary = pd.DataFrame({
    "sample": ["Train: 2011–2012", "Test: 2013"],
    "loans": [len(train_df), len(test_df)],
    "charge_off_rate": [
        100 * train_df["is_default"].mean(),
        100 * test_df["is_default"].mean(),
    ],
})

split_summary.round(2)

,sample,loans,charge_off_rate
0,Train: 2011–2012,75088,15.9
1,Test: 2013,134804,15.6


## 2. Preprocessing and logistic regression

The pipeline fills missing numeric values with the training-set median, standardizes numeric fields, and converts categories into indicator columns. Logistic regression then estimates each loan's probability of charge-off.

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

target = "is_default"
excluded = ["id", "issue_year", target]

X_train = train_df.drop(columns=excluded)
y_train = train_df[target]
X_test = test_df.drop(columns=excluded)
y_test = test_df[target]

categorical_features = X_train.select_dtypes(include="object").columns.tolist()
numeric_features = X_train.select_dtypes(exclude="object").columns.tolist()

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)
test_probability = model.predict_proba(X_test)[:, 1]

print("Model fitted successfully.")

Model fitted successfully.


## 3. Model performance

ROC-AUC evaluates how well the model ranks charged-off loans above fully paid loans. Average precision evaluates performance on the less-common charge-off outcome and is compared with the test-set charge-off rate as a no-skill baseline.

In [14]:
from sklearn.metrics import average_precision_score, roc_auc_score

metrics = pd.DataFrame({
    "metric": [
        "ROC-AUC",
        "Average precision",
        "No-skill precision baseline",
    ],
    "value": [
        roc_auc_score(y_test, test_probability),
        average_precision_score(y_test, test_probability),
        y_test.mean(),
    ],
})

metrics.round(4)

,metric,value
0,ROC-AUC,0.6733
1,Average precision,0.2673
2,No-skill precision baseline,0.1560


### Risk concentration

The test loans are divided into ten equal groups based on predicted risk. Decile 10 contains the highest-risk loans and Decile 1 the lowest-risk loans. This shows whether the score meaningfully concentrates actual charge-offs.

In [15]:
risk_results = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "predicted_probability": test_probability,
})

risk_results["risk_decile"] = (
    pd.qcut(risk_results["predicted_probability"], 10, labels=False) + 1
)

decile_summary = (
    risk_results
    .groupby("risk_decile")
    .agg(
        loans=("actual", "size"),
        charged_off_loans=("actual", "sum"),
        charge_off_rate=("actual", "mean"),
        average_predicted_risk=("predicted_probability", "mean"),
    )
    .reset_index()
    .sort_values("risk_decile", ascending=False)
)

decile_summary["charge_off_rate"] *= 100
decile_summary["average_predicted_risk"] *= 100
decile_summary["share_of_all_charge_offs"] = (
    100 * decile_summary["charged_off_loans"]
    / decile_summary["charged_off_loans"].sum()
)
decile_summary["cumulative_charge_off_capture"] = (
    decile_summary["share_of_all_charge_offs"].cumsum()
)

decile_summary.round(2)

,risk_decile,loans,charged_off_loans,charge_off_rate,average_predicted_risk,share_of_all_charge_offs,cumulative_charge_off_capture
9,10,13481,4404,32.67,37.69,20.95,20.95
8,9,13480,3367,24.98,27.56,16.02,36.96
7,8,13480,2766,20.52,22.29,13.16,50.12
6,7,13481,2307,17.11,18.97,10.97,61.09
5,6,13480,2043,15.16,16.58,9.72,70.81
4,5,13480,1768,13.12,14.57,8.41,79.22
3,4,13481,1514,11.23,12.70,7.20,86.42
2,3,13480,1222,9.07,10.78,5.81,92.23
1,2,13480,989,7.34,8.55,4.70,96.94
0,1,13481,644,4.78,4.97,3.06,100.00


## 4. Key model drivers

Permutation importance measures how much ROC-AUC declines when one original feature is randomly scrambled. Larger declines indicate that the model relied more heavily on that feature.

In [16]:
from sklearn.inspection import permutation_importance

importance_sample = X_test.sample(n=30000, random_state=42)
importance_target = y_test.loc[importance_sample.index]

permutation_result = permutation_importance(
    model,
    importance_sample,
    importance_target,
    scoring="roc_auc",
    n_repeats=3,
    random_state=42,
    n_jobs=-1,
)

feature_importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": permutation_result.importances_mean,
})

feature_importance = feature_importance.sort_values(
    "importance", ascending=False
).reset_index(drop=True)

feature_importance.head(10).round(4)

,feature,importance
0,term,0.0568
1,fico_score,0.0341
2,annual_inc,0.0268
3,loan_amnt,0.0164
4,inq_last_6mths,0.0144
5,purpose,0.0096
6,open_acc,0.0028
7,dti,0.0024
8,revol_util,0.0022
9,total_acc,0.0016


## 5. Tableau export

The scored 2013 loans and summary tables are exported for a business-facing Tableau dashboard. Loan IDs are used only for the merge and removed from the published dataset.

In [17]:
scored_loans = test_df.copy()
scored_loans["predicted_risk"] = test_probability
scored_loans["risk_decile"] = (
    pd.qcut(scored_loans["predicted_risk"], 10, labels=False) + 1
)

dashboard_fields = con.execute("""
    SELECT id, grade, sub_grade, int_rate, loan_status
    FROM loans_mature
""").df()

scored_loans = scored_loans.merge(dashboard_fields, on="id", how="left")
scored_loans = scored_loans.drop(columns="id")

outputs_dir = project_dir / "outputs"
outputs_dir.mkdir(exist_ok=True)

scored_loans.to_csv(outputs_dir / "tableau_loan_scores.csv", index=False)
decile_summary.to_csv(outputs_dir / "risk_decile_summary.csv", index=False)
feature_importance.to_csv(outputs_dir / "feature_importance.csv", index=False)
metrics.to_csv(outputs_dir / "model_metrics.csv", index=False)

print(f"Tableau files exported to: {outputs_dir}")

Tableau files exported to: /Users/meetshah/Documents/Codex/lendingclub-risk-analysis/outputs


## Conclusion

The interpretable baseline achieved meaningful out-of-time risk ranking without using LendingClub's assigned grade or interest rate. The highest-risk 30% of 2013 loans captured approximately half of all charge-offs. Loan term, FICO score, income, loan amount, recent credit inquiries, and purpose contributed most to the ranking. The model is intended for portfolio segmentation and analysis—not as a production lending decision system.